# Block 1: Document Normalization and ROI Extraction Framework

This notebook demonstrates **Block 1** of the Medical Document Understanding pipeline.

### Capabilities:
1. **Automatic Document Quad Detection & Homography Warp**: Corrects perspective and angle distortions from phone photographs or flatbed scans.
2. **Orientation Correction**: Detects and fixes 180° upside-down pages.
3. **Adaptive Template Matching**: Dynamically matches digital prints (`v0`, 124 checkboxes) or clinic prints (`v1`, 138 checkboxes).
4. **Sub-Pixel Anchor Snapping**: Snaps individual checkboxes to actual detected ink on the page, eliminating drift from paper folds or lighting shadows.
5. **High-Quality Crop Extraction & CLAHE Normalization**: Extracts every checkbox and handwriting field with both raw RGB and contrast-normalized crops.

## 1. Install Dependencies & Setup Environment

In [ ]:
# Install required packages (Google Colab compatible)
!pip install -q opencv-python-headless Pillow pydantic matplotlib

import os
import sys
import json
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Ensure current directory is on python path
if os.path.exists('block1'):
    os.chdir('block1')
sys.path.insert(0, os.path.abspath('.'))

from med_doc.normalization.pipeline import normalize_document
from med_doc.paths import DEFAULT_TEMPLATE, V1_TEMPLATE, SAMPLES_DIR

print("Environment initialized successfully!")
print(f"Working directory: {os.getcwd()}")

## 2. Load Sample or Upload Your Own Image

In [ ]:
# Set USE_UPLOAD = True to upload a file from your computer, or False to use the included sample
USE_UPLOAD = False

input_image_path = None

if USE_UPLOAD:
    from google.colab import files
    print("Please select an image file to upload:")
    uploaded = files.upload()
    if uploaded:
        input_image_path = list(uploaded.keys())[0]
        print(f"Uploaded: {input_image_path}")
else:
    # Use included sample
    sample_files = list(Path('samples').glob('*.png')) + list(Path('samples').glob('*.jpg'))
    if sample_files:
        input_image_path = str(sample_files[0])
        print(f"Using included sample: {input_image_path}")
    else:
        print("No sample files found in samples/ directory.")

if input_image_path:
    raw_img = Image.open(input_image_path)
    print(f"Image Size: {raw_img.size[0]} x {raw_img.size[1]} | Mode: {raw_img.mode}")
    
    plt.figure(figsize=(10, 8))
    plt.imshow(raw_img)
    plt.title(f"Input Image: {Path(input_image_path).name}")
    plt.axis('off')
    plt.show()

## 3. Run Block 1 Document Normalization

In [ ]:
print("Running Block 1 Normalization...")
result = normalize_document(raw_img, document_id=Path(input_image_path).stem)

print("\n=== Normalization Results ===")
print(f"Template Selected     : {result.extra.get('template_id')}")
print(f"Warp Method           : {result.warp_method}")
print(f"Orientation Correction: {result.orientation_degrees} deg")
print(f"Alignment Confidence  : {result.alignment_confidence:.3f}")
print(f"Extracted Checkboxes  : {len(result.checkbox_crops)}")
print(f"Extracted Handwriting : {len(result.handwriting_crops)}")

## 4. Visual Inspection (Overlay & Bounding Boxes)

In [ ]:
plt.figure(figsize=(18, 14))
plt.subplot(1, 2, 1)
plt.imshow(result.canonical_canvas)
plt.title(f"Rectified Canonical Canvas ({result.canonical_canvas.shape[1]}x{result.canonical_canvas.shape[0]})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(result.debug_overlay)
plt.title("Debug Overlay (Green=Checkboxes, Blue=Handwriting ROIs)")
plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Crop Gallery (Inspect Extracted ROIs)

In [ ]:
# Display first 12 checkbox crops
cb_items = list(result.checkbox_crops.items())[:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, (fid, crop) in zip(axes.ravel(), cb_items):
    ax.imshow(crop.normalized_image)
    ax.set_title(f"{fid}\nq={crop.quality_score:.2f}", fontsize=9)
    ax.axis('off')
plt.suptitle("Sample Extracted Checkbox Crops (CLAHE Normalized)", fontsize=14)
plt.tight_layout()
plt.show()

# Display handwriting crops
hw_items = list(result.handwriting_crops.items())[:4]
if hw_items:
    fig, axes = plt.subplots(len(hw_items), 1, figsize=(10, 2.5 * len(hw_items)))
    if len(hw_items) == 1:
        axes = [axes]
    for ax, (fid, crop) in zip(axes, hw_items):
        ax.imshow(crop.normalized_image)
        ax.set_title(f"{fid} | shape: {crop.raw_image.shape[:2]} | quality: {crop.quality_score:.2f}")
        ax.axis('off')
    plt.suptitle("Sample Handwriting ROIs", fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Export and Download All Crops as a ZIP File

In [ ]:
import shutil
import zipfile

output_dir = Path("outputs/colab_run")
output_dir.mkdir(parents=True, exist_ok=True)

# Save crops
for fid, crop in list(result.checkbox_crops.items()) + list(result.handwriting_crops.items()):
    raw_bgr = cv2.cvtColor(crop.raw_image, cv2.COLOR_RGB2BGR) if crop.raw_image.ndim == 3 else crop.raw_image
    norm_bgr = cv2.cvtColor(crop.normalized_image, cv2.COLOR_RGB2BGR) if crop.normalized_image.ndim == 3 else crop.normalized_image
    cv2.imwrite(str(output_dir / f"{fid}_raw.png"), raw_bgr)
    cv2.imwrite(str(output_dir / f"{fid}_norm.png"), norm_bgr)

# Save debug overlay
cv2.imwrite(str(output_dir / "overlay.jpg"), cv2.cvtColor(result.debug_overlay, cv2.COLOR_RGB2BGR))

# Zip everything
zip_filename = "block1_normalization_results.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), output_dir))

print(f"ZIP package created: {zip_filename} ({os.path.getsize(zip_filename)} bytes)")

# If running in Google Colab, trigger automatic download
try:
    from google.colab import files
    files.download(zip_filename)
    print("Download started!")
except Exception as e:
    print(f"(Local execution) ZIP file saved to {zip_filename}")